# Convolution Layer (Conv2d)

## Overview
This notebook covers the practical implementation of convolutional layers, including parameters like kernel size, stride, padding, and channels. We'll see how these affect feature map dimensions and learn to build Conv2d layers.

## Learning Objectives
- Understand Conv2d layer parameters
- Calculate feature map dimensions
- Learn about stride and padding effects
- Implement convolutional layers in PyTorch

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import math

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Conv2d Layer Parameters

The `nn.Conv2d` layer has several important parameters:

### Required Parameters:
- **in_channels**: Number of input channels (e.g., 3 for RGB)
- **out_channels**: Number of output feature maps
- **kernel_size**: Size of the convolution kernel

### Optional Parameters:
- **stride**: Step size when sliding the kernel (default: 1)
- **padding**: Zero-padding added to input (default: 0)
- **bias**: Whether to add bias term (default: True)
- **dilation**: Spacing between kernel elements (default: 1)

In [3]:
# Basic Conv2d layer examples
print("Basic Conv2d Layer Examples:")
print("-" * 30)

# Example 1: RGB to 32 feature maps
conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)
print(f"Conv1: {conv1}")
print(f"Parameters: {sum(p.numel() for p in conv1.parameters()):,}")
print()

# Example 2: With stride and padding
conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=2, padding=2)
print(f"Conv2: {conv2}")
print(f"Parameters: {sum(p.numel() for p in conv2.parameters()):,}")
print()

# Example 3: 1x1 convolution (common in modern architectures)
conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=1)
print(f"Conv3: {conv3}")
print(f"Parameters: {sum(p.numel() for p in conv3.parameters()):,}")

Basic Conv2d Layer Examples:
------------------------------
Conv1: Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
Parameters: 896

Conv2: Conv2d(32, 64, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
Parameters: 51,264

Conv3: Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))
Parameters: 8,320


## Feature Map Dimensions

Understanding how input dimensions change through convolution is crucial for building CNNs.

### Formula:
```
Output Size = (Input Size - Kernel Size + 2×Padding) / Stride + 1
```

### Input Format:
- **Input**: (Batch, Channels, Height, Width)
- **Filter**: (Out_Channels, In_Channels, Kernel_Height, Kernel_Width)
- **Output**: (Batch, Out_Channels, New_Height, New_Width)

In [12]:
def calculate_output_size(input_size, kernel_size, stride=1, padding=0):
    """Calculate output size after convolution"""
    return (input_size - kernel_size + 2 * padding) // stride + 1

def demonstrate_conv_dimensions():
    """Demonstrate how different parameters affect output dimensions"""

    print("Feature Map Dimension Changes:")
    print("=" * 50)

    # Test different configurations
    configs = [
        {'input': 32, 'kernel': 3, 'stride': 1, 'padding': 0, 'name': 'Basic 3x3'},
        {'input': 32, 'kernel': 3, 'stride': 1, 'padding': 1, 'name': 'Padding 1'},
        {'input': 32, 'kernel': 3, 'stride': 2, 'padding': 1, 'name': 'Padding 1 + Stride 2'},
        {'input': 32, 'kernel': 5, 'stride': 1, 'padding': 2, 'name': 'Padding 1 + Stride 2 + 5x5 same'},
        {'input': 32, 'kernel': 1, 'stride': 1, 'padding': 0, 'name': '1x1 conv'},
    ]

    for config in configs:
        output_size = calculate_output_size(
            config['input'], config['kernel'],
            config['stride'], config['padding']
        )

        print(f"{config['name']:40} | "
              f"Input: {config['input']:2}×{config['input']:2} | "
              f"Kernel: {config['kernel']}×{config['kernel']} | "
              f"Stride: {config['stride']} | "
              f"Padding: {config['padding']} | "
              f"Output: {output_size:2}×{output_size:2}")

    print("\nKey Observations:")
    print("- Same padding: output size = input size (when stride=1)")
    print("- Stride=2: output size ≈ input size / 2")
    print("- 1×1 conv: preserves spatial dimensions")

demonstrate_conv_dimensions()

Feature Map Dimension Changes:
Basic 3x3                                | Input: 32×32 | Kernel: 3×3 | Stride: 1 | Padding: 0 | Output: 30×30
Padding 1                                | Input: 32×32 | Kernel: 3×3 | Stride: 1 | Padding: 1 | Output: 32×32
Padding 1 + Stride 2                     | Input: 32×32 | Kernel: 3×3 | Stride: 2 | Padding: 1 | Output: 16×16
Padding 1 + Stride 2 + 5x5 same          | Input: 32×32 | Kernel: 5×5 | Stride: 1 | Padding: 2 | Output: 32×32
1x1 conv                                 | Input: 32×32 | Kernel: 1×1 | Stride: 1 | Padding: 0 | Output: 32×32

Key Observations:
- Same padding: output size = input size (when stride=1)
- Stride=2: output size ≈ input size / 2
- 1×1 conv: preserves spatial dimensions


## Practical Examples with Real Tensors

In [11]:
# Create sample input tensor (batch_size=1, channels=3, height=32, width=32)
input_tensor = torch.randn(1, 3, 32, 32)
print(f"Input tensor shape: {input_tensor.shape}")
print(f"Format: (Batch, Channels, Height, Width)")
print()

# Test different convolution configurations
convolutions = [
    nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=0),  # No padding
    nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),  # Same padding
    nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),  # Stride 2
    nn.Conv2d(3, 16, kernel_size=5, stride=1, padding=2),  # 5x5 kernel
    nn.Conv2d(3, 16, kernel_size=1, stride=1, padding=0),  # 1x1 conv
]

names = ['No padding', 'Padding 1', 'Padding 1 + Stride 2', 'Padding 1 + Stride 2 + 5x5 kernel', '1x1 conv']

print("Convolution Results:")
print("-" * 60)
for conv, name in zip(convolutions, names):
    output = conv(input_tensor)
    params = sum(p.numel() for p in conv.parameters())
    print(f"{name:40} | Input: {list(input_tensor.shape)} → Output: {list(output.shape)} | Params: {params:,}")

Input tensor shape: torch.Size([1, 3, 32, 32])
Format: (Batch, Channels, Height, Width)

Convolution Results:
------------------------------------------------------------
No padding                               | Input: [1, 3, 32, 32] → Output: [1, 16, 30, 30] | Params: 448
Padding 1                                | Input: [1, 3, 32, 32] → Output: [1, 16, 32, 32] | Params: 448
Padding 1 + Stride 2                     | Input: [1, 3, 32, 32] → Output: [1, 16, 16, 16] | Params: 448
Padding 1 + Stride 2 + 5x5 kernel        | Input: [1, 3, 32, 32] → Output: [1, 16, 32, 32] | Params: 1,216
1x1 conv                                 | Input: [1, 3, 32, 32] → Output: [1, 16, 32, 32] | Params: 64


## Key Takeaways

### Conv2d Parameters:
- **in_channels**: Input depth
- **out_channels**: Number of filters/feature maps
- **kernel_size**: Filter size (typically 3×3 or 5×5)
- **stride**: Controls downsampling
- **padding**: Controls output size

### Feature Map Dimensions:
```
Output Size = (Input - Kernel + 2×Padding) / Stride + 1
```

### Common Patterns:
- **Same padding**: `padding = kernel_size // 2` (preserves size when stride=1)
- **Downsampling**: Use `stride=2` to halve spatial dimensions
- **Channel expansion**: Increase channels as spatial size decreases
- **1×1 convolutions**: Change channel depth without affecting spatial size

### Best Practices:
- Use 3×3 kernels (efficient and expressive)
- Add BatchNorm after convolution
- Use ReLU activation
- Gradually increase channels while decreasing spatial size
- End with global pooling instead of large fully connected layers